# Exploratory Data Analysis (EDA)

Template for quick data inspection.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report



: 

## Load Data

In [ ]:
from pathlib import Path

# 1) Locate RAND HRS file (local, Colab workspace, or Google Drive)
file_name = "randhrs1992_2022v1.dta"

# Optional: mount Google Drive automatically when running in Colab
try:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

candidate_paths = [
    Path("content/data/randhrs1992_2022v1_STATA/randhrs1992_2022v1.dta"),
    Path("../content/data/randhrs1992_2022v1_STATA/randhrs1992_2022v1.dta"),
    Path("/content/hea_hackathon/content/data/randhrs1992_2022v1_STATA/randhrs1992_2022v1.dta"),
    Path("/content/data/randhrs1992_2022v1_STATA/randhrs1992_2022v1.dta"),
    Path("/content/drive/MyDrive/hea_hackathon/content/data/randhrs1992_2022v1_STATA/randhrs1992_2022v1.dta"),
    Path("/content/drive/MyDrive/randhrs1992_2022v1.dta"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)

# Fallback: recursive search in common roots
if data_path is None:
    search_roots = [Path.cwd(), Path("/content"), Path("/content/drive"), Path("/content/drive/MyDrive"), Path("/kaggle/input")]
    search_roots = [p for p in search_roots if p.exists()]
    found = []
    for root in search_roots:
        try:
            matches = list(root.rglob(file_name))
            if matches:
                found.extend(matches)
                break
        except (PermissionError, OSError):
            continue
    if found:
        data_path = found[0]

if data_path is None:
    tried = "\n".join([f"- {str(p)}" for p in candidate_paths])
    raise FileNotFoundError(
        "Could not find randhrs1992_2022v1.dta. Checked these paths:\n"
        f"{tried}\n\n"
        f"Current working directory: {Path.cwd()}"
    )

print(f"Using dataset: {data_path}")

id_col = "hhidpn"
static_demo_cols = ["ragender", "raracem", "rahispan"]
feature_suffixes = [
    "agey_e",
    "cesd",
    "bmi",
    "mobila",
    "shltc",
    "smoken",
    "drink",
    "conde",
    "shlt",
    "adl6a",
    "iadl5a",
    "walks",
    "drinkd",
    "drinkn",
    "work",
    "mstat",
    "finr",
    "urbrur",
    "proxy",
]
disease_suffixes = ["hibpe", "diabe", "hearte", "stroke", "arthre"]

with pd.io.stata.StataReader(str(data_path)) as reader:
    available = set(reader.variable_labels().keys())

selected_static_demo_cols = [c for c in static_demo_cols if c in available]
selected_feature_suffixes = [s for s in feature_suffixes if f"r13{s}" in available and f"r14{s}" in available]
selected_diseases = [d for d in disease_suffixes if f"r13{d}" in available and f"r14{d}" in available and f"r15{d}" in available]

if not selected_diseases:
    raise ValueError("No complete R13/R14/R15 disease triplets were found.")

wave_cols = []
for w in [13, 14]:
    wave_cols.extend([f"r{w}{s}" for s in selected_feature_suffixes])
for w in [13, 14, 15]:
    wave_cols.extend([f"r{w}{d}" for d in selected_diseases])

required_cols = [id_col] + selected_static_demo_cols + wave_cols
required_cols = list(dict.fromkeys(required_cols))

with pd.io.stata.StataReader(str(data_path)) as reader:
    df = reader.read(columns=required_cols, convert_categoricals=False)

# Keep rows with at least one informative signal
signal_cols = [c for c in required_cols if c != id_col]
df = df.dropna(subset=signal_cols, how="all").reset_index(drop=True)

# Baseline reference target (R15 prevalence)
r15_target_cols = [f"r15{d}" for d in selected_diseases]
for c in r15_target_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

target_col = "target_any_r15_condition"
df[target_col] = (df[r15_target_cols].fillna(0) > 0).any(axis=1).astype(int)

print(f"Loaded: {data_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Selected diseases: {selected_diseases}")
print(f"Static demo features: {len(selected_static_demo_cols)}")
print(f"Wave feature suffixes (R13/R14): {len(selected_feature_suffixes)}")
print(f"R15 targets: {r15_target_cols}")
print(f"Positive rate ({target_col}): {df[target_col].mean():.3f}")

df.head()



In [ ]:
# Quick sanity checks
r13_targets = [f"r13{d}" for d in selected_diseases]
r14_targets = [f"r14{d}" for d in selected_diseases]
r15_targets = [f"r15{d}" for d in selected_diseases]

print("Selected wave feature suffixes:", selected_feature_suffixes)
print("\nMissingness by wave targets:")
print(df[r13_targets + r14_targets + r15_targets].isna().mean().sort_values().to_string())

print("\nPrevalence by wave:")
for cols in [r13_targets, r14_targets, r15_targets]:
    print("-", cols[0][:3], "mean prevalence")
    print(df[cols].mean().sort_values(ascending=False).to_string())



In [ ]:
# Build per-disease incidence datasets using transitions: 13->14 and 14->15
import numpy as np
import pandas as pd

binary_feature_suffixes = {"smoken", "drink", "proxy"}
categorical_feature_names = {
    "ragender",
    "raracem",
    "rahispan",
    "smoken_prev",
    "drink_prev",
    "work_prev",
    "mstat_prev",
    "finr_prev",
    "urbrur_prev",
    "proxy_prev",
    "shlt_prev",
    "transition_prev_wave",
}

# These are treated as continuous/count-like for delta engineering
numeric_delta_suffixes = {
    "agey_e", "cesd", "bmi", "mobila", "shltc", "conde", "shlt", "adl6a", "iadl5a", "walks", "drinkd", "drinkn"
}

min_observed_ratio = 0.35

datasets = {}
summary_rows = []

for disease in selected_diseases:
    chunks = []

    for prev_w, next_w in [(13, 14), (14, 15)]:
        prev_target_col = f"r{prev_w}{disease}"
        next_target_col = f"r{next_w}{disease}"

        prev_feature_cols = [f"r{prev_w}{s}" for s in selected_feature_suffixes]
        prev_hist_cols = [f"r{prev_w}{d}" for d in selected_diseases]

        # Extra longitudinal context only for 14->15 (safe: uses past wave 13)
        prev2_feature_cols = []
        prev2_hist_cols = []
        if prev_w == 14:
            prev2_feature_cols = [f"r13{s}" for s in selected_feature_suffixes if f"r13{s}" in df.columns]
            prev2_hist_cols = [f"r13{d}" for d in selected_diseases if f"r13{d}" in df.columns]

        cols_needed = (
            [id_col]
            + selected_static_demo_cols
            + prev_feature_cols
            + prev_hist_cols
            + prev2_feature_cols
            + prev2_hist_cols
            + [prev_target_col, next_target_col]
        )
        cols_needed = list(dict.fromkeys(cols_needed))

        tmp = df.loc[:, cols_needed].copy()

        rename_map = {}
        for s in selected_feature_suffixes:
            k_prev = f"r{prev_w}{s}"
            if k_prev in tmp.columns:
                rename_map[k_prev] = f"{s}_prev"
            if prev_w == 14:
                k_prev2 = f"r13{s}"
                if k_prev2 in tmp.columns:
                    rename_map[k_prev2] = f"{s}_prev2"

        for d in selected_diseases:
            k_hist_prev = f"r{prev_w}{d}"
            if k_hist_prev in tmp.columns:
                rename_map[k_hist_prev] = f"hist_{d}_prev"
            if prev_w == 14:
                k_hist_prev2 = f"r13{d}"
                if k_hist_prev2 in tmp.columns:
                    rename_map[k_hist_prev2] = f"hist_{d}_prev2"

        rename_map[prev_target_col] = "label_hist_prev"
        rename_map[next_target_col] = "label_next"

        tmp = tmp.rename(columns=rename_map)
        tmp["transition_prev_wave"] = prev_w

        # Numeric coercion
        for c in tmp.columns:
            if c == id_col:
                continue
            col = tmp[c]
            if isinstance(col, pd.DataFrame):
                col = col.iloc[:, 0]
            tmp[c] = pd.to_numeric(col, errors="coerce")

        # Clean invalid encodings in binary fields
        for bs in binary_feature_suffixes:
            for col in [f"{bs}_prev", f"{bs}_prev2"]:
                if col in tmp.columns:
                    tmp.loc[~tmp[col].isin([0, 1]), col] = np.nan

        hist_cols = [c for c in tmp.columns if c.startswith("hist_")]
        for hc in hist_cols + ["label_hist_prev", "label_next"]:
            tmp.loc[~tmp[hc].isin([0, 1]), hc] = np.nan

        # Most features should be non-negative; keep shltc as signed scale
        for c in tmp.columns:
            if c in {id_col, "shltc_prev", "shltc_prev2"}:
                continue
            if c.endswith("_prev") or c.endswith("_prev2") or c in selected_static_demo_cols or c.startswith("hist_"):
                tmp.loc[tmp[c] < 0, c] = np.nan

        valid = tmp["label_hist_prev"].isin([0, 1]) & tmp["label_next"].isin([0, 1])
        at_risk = tmp["label_hist_prev"] == 0
        tmp = tmp.loc[valid & at_risk].copy()

        # Drop very sparse rows (reduces heavy imputation noise)
        model_candidate_cols = [c for c in tmp.columns if c not in {id_col, "label_hist_prev", "label_next"}]
        tmp["observed_ratio_prev"] = tmp[model_candidate_cols].notna().mean(axis=1)
        tmp = tmp.loc[tmp["observed_ratio_prev"] >= min_observed_ratio].copy()

        tmp["y"] = (tmp["label_next"] == 1).astype("int8")

        # Engineered signals
        other_hist = [f"hist_{d}_prev" for d in selected_diseases if d != disease and f"hist_{d}_prev" in tmp.columns]
        if other_hist:
            tmp["other_disease_count_prev"] = tmp[other_hist].sum(axis=1, min_count=1)

        other_hist2 = [f"hist_{d}_prev2" for d in selected_diseases if d != disease and f"hist_{d}_prev2" in tmp.columns]
        if other_hist2:
            tmp["other_disease_count_prev2"] = tmp[other_hist2].sum(axis=1, min_count=1)

        if "bmi_prev" in tmp.columns and "agey_e_prev" in tmp.columns:
            tmp["bmi_age_interaction"] = tmp["bmi_prev"] * tmp["agey_e_prev"]

        if "smoken_prev" in tmp.columns and "drink_prev" in tmp.columns:
            tmp["smoke_drink_interaction"] = tmp["smoken_prev"] * tmp["drink_prev"]

        # Delta between prev and prev2 (available mostly in 14->15 rows)
        for s in selected_feature_suffixes:
            if s not in numeric_delta_suffixes:
                continue
            c1 = f"{s}_prev"
            c2 = f"{s}_prev2"
            if c1 in tmp.columns and c2 in tmp.columns:
                tmp[f"delta_{s}_prev"] = tmp[c1] - tmp[c2]

        chunks.append(tmp)

    full = pd.concat(chunks, ignore_index=True)

    drop_cols = {id_col, "label_hist_prev", "label_next", "y"}
    X_cols = [c for c in full.columns if c not in drop_cols]
    X_d = full[X_cols].copy()

    # Remove constant columns
    non_constant = [c for c in X_d.columns if X_d[c].nunique(dropna=True) > 1]
    X_d = X_d[non_constant]

    y_d = full["y"].copy()
    groups_d = full[id_col].copy()
    transition_wave_d = full["transition_prev_wave"].copy()

    datasets[disease] = {
        "X": X_d,
        "y": y_d,
        "groups": groups_d,
        "transition_wave": transition_wave_d,
        "categorical_cols": [c for c in categorical_feature_names if c in X_d.columns],
    }

    summary_rows.append(
        {
            "disease": disease,
            "rows": int(len(full)),
            "unique_people": int(groups_d.nunique()),
            "positives": int(y_d.sum()),
            "positive_rate": float(y_d.mean()),
            "n_features": int(X_d.shape[1]),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("positive_rate", ascending=False).reset_index(drop=True)
print("Per-disease transition datasets (13->14 + 14->15):")
summary_df



## Train + Validation (Imbalanced Multi-Disease)


In [ ]:
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, fbeta_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

# Optional extra models for ensemble
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

if not datasets:
    raise ValueError("datasets is empty. Run the previous cell first.")

# Finer search near low thresholds for rare outcomes
threshold_grid = np.unique(np.concatenate([
    np.linspace(0.002, 0.08, 157),
    np.linspace(0.081, 0.98, 180),
]))

# Soft precision guards (hackathon still prioritizes F2)
min_precision_floor = {
    "hibpe": 0.07,
    "arthre": 0.07,
    "hearte": 0.05,
    "diabe": 0.05,
    "stroke": 0.03,
}

results = []
artifacts = {}
eval_payloads = {}

param_candidates = [
    {
        "n_estimators": 600,
        "learning_rate": 0.03,
        "num_leaves": 63,
        "min_child_samples": 10,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
    {
        "n_estimators": 900,
        "learning_rate": 0.02,
        "num_leaves": 127,
        "min_child_samples": 5,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_alpha": 0.0,
        "reg_lambda": 0.0,
    },
]

seed_list = [42, 52]

for disease, pack in datasets.items():
    X_d = pack["X"].copy()
    y_d = pack["y"].copy()
    groups = pack["groups"].copy()
    transition_wave = pack["transition_wave"].copy()
    cat_cols = pack.get("categorical_cols", [])

    if y_d.nunique() < 2 or int(y_d.sum()) < 30:
        print(f"Skipping {disease}: not enough positive samples for stable training.")
        continue

    # 1) Drop very sparse features
    miss = X_d.isna().mean()
    keep_cols = miss[miss <= 0.60].index.tolist()
    X_d = X_d[keep_cols].copy()

    # Keep pre-model view for fairness/explainability payloads
    X_pre_model = X_d.copy()

    # 2) Add missing indicators and clip outliers on numeric features
    num_cols = X_d.select_dtypes(include=["number"]).columns.tolist()
    for c in num_cols:
        X_d[f"{c}__na"] = X_d[c].isna().astype("int8")
        q01, q99 = X_d[c].quantile([0.01, 0.99])
        if pd.notna(q01) and pd.notna(q99):
            X_d[c] = X_d[c].clip(q01, q99)

    # 3) One-hot only available categorical columns
    cat_cols = [c for c in cat_cols if c in X_d.columns]
    X_d = pd.get_dummies(X_d, columns=cat_cols, dummy_na=True)

    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.4, random_state=42)
    train_idx, temp_idx = next(gss1.split(X_d, y_d, groups=groups))

    X_train = X_d.iloc[train_idx]
    y_train = y_d.iloc[train_idx]
    trans_train = transition_wave.iloc[train_idx]

    X_temp = X_d.iloc[temp_idx]
    y_temp = y_d.iloc[temp_idx]
    g_temp = groups.iloc[temp_idx]

    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    val_rel_idx, test_rel_idx = next(gss2.split(X_temp, y_temp, groups=g_temp))

    X_val, y_val = X_temp.iloc[val_rel_idx], y_temp.iloc[val_rel_idx]
    X_test, y_test = X_temp.iloc[test_rel_idx], y_temp.iloc[test_rel_idx]

    imputer = SimpleImputer(strategy="median")
    X_train_i = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_val_i = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)
    X_test_i = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

    pos_train = int(y_train.sum())
    neg_train = int(len(y_train) - pos_train)
    spw_raw = max(1.0, neg_train / max(pos_train, 1))

    pos_weight_modes = {
        "none": 1.0,
        "sqrt_spw": float(np.sqrt(spw_raw)),
        "capped_spw": float(min(spw_raw, 8.0)),
    }
    recency_factors = [1.0, 1.2, 1.4]

    best_bundle = None

    for candidate in param_candidates:
        for w_name, pos_w in pos_weight_modes.items():
            for recency_factor in recency_factors:
                val_pred_acc = np.zeros(len(X_val_i), dtype=float)
                test_pred_acc = np.zeros(len(X_test_i), dtype=float)
                models = []

                class_w = np.where(y_train.values == 1, pos_w, 1.0)
                recency_w = np.where(trans_train.values == 14, recency_factor, 1.0)
                sample_w = class_w * recency_w

                for seed in seed_list:
                    model = LGBMClassifier(
                        objective="binary",
                        random_state=seed,
                        n_jobs=-1,
                        force_row_wise=True,
                        scale_pos_weight=1.0,
                        verbosity=-1,
                        **candidate,
                    )
                    model.fit(X_train_i, y_train, sample_weight=sample_w)
                    val_pred_acc += model.predict_proba(X_val_i)[:, 1] / len(seed_list)
                    test_pred_acc += model.predict_proba(X_test_i)[:, 1] / len(seed_list)
                    models.append(model)

                f2_grid = [fbeta_score(y_val, (val_pred_acc >= t).astype(int), beta=2, zero_division=0) for t in threshold_grid]

                bundle = {
                    "models": models,
                    "val_pred": val_pred_acc,
                    "test_pred": test_pred_acc,
                    "val_f2": float(max(f2_grid)),
                    "val_pr": float(average_precision_score(y_val, val_pred_acc)),
                    "val_roc": float(roc_auc_score(y_val, val_pred_acc)),
                    "params": candidate,
                    "weight_mode": w_name,
                    "pos_w": float(pos_w),
                    "recency_factor": float(recency_factor),
                    "spw_raw": float(spw_raw),
                    "sample_w": sample_w,
                }

                if best_bundle is None:
                    best_bundle = bundle
                else:
                    old = (best_bundle["val_f2"], best_bundle["val_pr"], best_bundle["val_roc"])
                    new = (bundle["val_f2"], bundle["val_pr"], bundle["val_roc"])
                    if new > old:
                        best_bundle = bundle

    # Optional ensemble with XGBoost / CatBoost
    pred_val_dict = {"lgb": best_bundle["val_pred"]}
    pred_test_dict = {"lgb": best_bundle["test_pred"]}
    extra_models = {}

    if HAS_XGB:
        xgb = XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            n_jobs=-1,
            verbosity=0,
            random_state=42,
        )
        xgb.fit(X_train_i, y_train, sample_weight=best_bundle["sample_w"])
        pred_val_dict["xgb"] = xgb.predict_proba(X_val_i)[:, 1]
        pred_test_dict["xgb"] = xgb.predict_proba(X_test_i)[:, 1]
        extra_models["xgb"] = xgb

    if HAS_CAT:
        cat = CatBoostClassifier(
            iterations=500,
            learning_rate=0.03,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            verbose=False,
            random_seed=42,
        )
        cat.fit(X_train_i, y_train, sample_weight=best_bundle["sample_w"])
        pred_val_dict["cat"] = cat.predict_proba(X_val_i)[:, 1]
        pred_test_dict["cat"] = cat.predict_proba(X_test_i)[:, 1]
        extra_models["cat"] = cat

    model_names = list(pred_val_dict.keys())
    weight_grid = np.linspace(0.0, 1.0, 11)

    best_mix = None
    if len(model_names) == 1:
        best_mix = {"weights": {model_names[0]: 1.0}}
    else:
        from itertools import product

        for ws in product(weight_grid, repeat=len(model_names)):
            if abs(sum(ws) - 1.0) > 1e-9:
                continue
            w_map = {name: float(w) for name, w in zip(model_names, ws)}
            if w_map.get("lgb", 0.0) < 0.4:
                continue

            val_blend = np.zeros(len(X_val_i), dtype=float)
            for name, w in w_map.items():
                val_blend += w * pred_val_dict[name]

            f2_grid = [fbeta_score(y_val, (val_blend >= t).astype(int), beta=2, zero_division=0) for t in threshold_grid]
            score = (
                float(max(f2_grid)),
                float(average_precision_score(y_val, val_blend)),
                float(roc_auc_score(y_val, val_blend)),
            )

            candidate_mix = {"weights": w_map, "score": score}
            if best_mix is None or candidate_mix["score"] > best_mix["score"]:
                best_mix = candidate_mix

    final_weights = best_mix["weights"]

    p_val_final = np.zeros(len(X_val_i), dtype=float)
    p_test_final = np.zeros(len(X_test_i), dtype=float)
    for name, w in final_weights.items():
        p_val_final += w * pred_val_dict[name]
        p_test_final += w * pred_test_dict[name]

    # Probability calibration on validation predictions
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p_val_final, y_val)
    p_val_cal = iso.predict(p_val_final)
    p_test_cal = iso.predict(p_test_final)

    # Threshold selection: maximize F2 with soft precision guard
    min_prec = min_precision_floor.get(disease, 0.05)
    best_thr = None
    best_tuple = None

    for t in threshold_grid:
        yv_hat = (p_val_cal >= t).astype(int)
        f2 = float(fbeta_score(y_val, yv_hat, beta=2, zero_division=0))
        prec = float(precision_score(y_val, yv_hat, zero_division=0))

        # soft penalty only if precision is below floor
        penalty = 0.0 if prec >= min_prec else 0.02 * ((min_prec - prec) / max(min_prec, 1e-6))
        score = f2 - penalty
        candidate_tuple = (score, f2, prec)

        if best_tuple is None or candidate_tuple > best_tuple:
            best_tuple = candidate_tuple
            best_thr = float(t)

    thr_final = float(best_thr)
    y_pred = (p_test_cal >= thr_final).astype(int)

    # Keep evaluation payload for fairness/explainability
    meta_cols = [c for c in ["ragender", "raracem", "rahispan", "transition_prev_wave"] if c in X_pre_model.columns]
    X_test_meta = X_pre_model.loc[X_test.index, meta_cols].copy() if meta_cols else pd.DataFrame(index=X_test.index)
    X_test_raw = X_pre_model.loc[X_test.index].copy()

    eval_payloads[disease] = {
        "y_test": y_test.reset_index(drop=True),
        "p_test": pd.Series(p_test_cal).reset_index(drop=True),
        "y_pred": pd.Series(y_pred).reset_index(drop=True),
        "X_test_meta": X_test_meta.reset_index(drop=True),
        "X_test_raw": X_test_raw.reset_index(drop=True),
        "X_test_model": X_test_i.reset_index(drop=True),
        "threshold": float(thr_final),
    }

    results.append(
        {
            "disease": disease,
            "n_train": int(len(y_train)),
            "n_test": int(len(y_test)),
            "pos_rate_test": float(y_test.mean()),
            "spw_raw": float(best_bundle["spw_raw"]),
            "pos_weight_mode": best_bundle["weight_mode"],
            "pos_weight_value": float(best_bundle["pos_w"]),
            "recency_factor": float(best_bundle["recency_factor"]),
            "threshold": float(thr_final),
            "F2": float(fbeta_score(y_test, y_pred, beta=2, zero_division=0)),
            "PR-AUC": float(average_precision_score(y_test, p_test_cal)),
            "ROC-AUC": float(roc_auc_score(y_test, p_test_cal)),
            "Recall": float(recall_score(y_test, y_pred, zero_division=0)),
            "Precision": float(precision_score(y_test, y_pred, zero_division=0)),
            "val_F2": float(fbeta_score(y_val, (p_val_cal >= thr_final).astype(int), beta=2, zero_division=0)),
            "val_PR-AUC": float(average_precision_score(y_val, p_val_cal)),
            "val_ROC-AUC": float(roc_auc_score(y_val, p_val_cal)),
            "n_features_final": int(X_d.shape[1]),
            "blend": str(final_weights),
        }
    )

    artifacts[disease] = {
        "lgb_models": best_bundle["models"],
        "extra_models": extra_models,
        "imputer": imputer,
        "calibrator": iso,
        "features": list(X_d.columns),
        "threshold": float(thr_final),
        "params": best_bundle["params"],
        "pos_weight_mode": best_bundle["weight_mode"],
        "pos_weight_value": float(best_bundle["pos_w"]),
        "recency_factor": float(best_bundle["recency_factor"]),
        "spw_raw": float(best_bundle["spw_raw"]),
        "blend": final_weights,
    }

metrics_df = pd.DataFrame(results).sort_values("F2", ascending=False).reset_index(drop=True)
print("Per-disease metrics on test set:")
print(metrics_df.to_string(index=False))

if not metrics_df.empty:
    macro = metrics_df[["F2", "PR-AUC", "ROC-AUC", "Recall", "Precision"]].mean()
    print("\nMacro average:")
    print(macro.to_string())



## Metrics Export


In [ ]:
# Full metrics view + export for submission tracking
from pathlib import Path
import pandas as pd

if 'metrics_df' not in globals() or metrics_df.empty:
    raise ValueError('metrics_df is missing or empty. Run training cell first.')

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

metrics_df = metrics_df.sort_values(['F2', 'PR-AUC'], ascending=False).reset_index(drop=True)
metrics_df['PR_lift_vs_prevalence'] = metrics_df['PR-AUC'] / metrics_df['pos_rate_test'].clip(lower=1e-6)

print(metrics_df.to_string(index=False))
print('')
print('Macro:')
print(metrics_df[['F2', 'PR-AUC', 'ROC-AUC', 'Recall', 'Precision']].mean().to_string())

out_candidates = [
    Path('/content/drive/MyDrive/hea_metrics_latest.csv'),
    Path('hea_metrics_latest.csv'),
]

out_path = None
for p in out_candidates:
    try:
        if p.parent and str(p.parent) != '.':
            p.parent.mkdir(parents=True, exist_ok=True)
        metrics_df.to_csv(p, index=False)
        out_path = p
        break
    except Exception:
        continue

if out_path is None:
    raise OSError('Could not write metrics CSV to any candidate path.')

print(f'Saved metrics to: {out_path}')



## Fairness + Leakage + Explainability


In [ ]:
# Fairness audit (group-wise TPR/FPR/precision)
import numpy as np
import pandas as pd

if 'eval_payloads' not in globals() or not eval_payloads:
    raise ValueError('Run training cell first: eval_payloads is missing.')

protected_cols = ['ragender', 'raracem', 'rahispan', 'transition_prev_wave']
min_group_n = 80

fair_rows = []
gap_rows = []

for disease, payload in eval_payloads.items():
    base = pd.DataFrame({
        'y_true': payload['y_test'].values,
        'y_pred': payload['y_pred'].values,
        'p_test': payload['p_test'].values,
    })
    meta = payload.get('X_test_meta', pd.DataFrame())
    eval_df = pd.concat([base, meta.reset_index(drop=True)], axis=1)

    for attr in protected_cols:
        if attr not in eval_df.columns:
            continue

        counts = eval_df[attr].value_counts(dropna=False)
        valid_groups = counts[counts >= min_group_n].index.tolist()
        if len(valid_groups) < 2:
            continue

        per_group = []
        for g in valid_groups:
            sub = eval_df[eval_df[attr] == g]
            y_true = sub['y_true'].values
            y_pred = sub['y_pred'].values

            tp = int(((y_true == 1) & (y_pred == 1)).sum())
            fp = int(((y_true == 0) & (y_pred == 1)).sum())
            tn = int(((y_true == 0) & (y_pred == 0)).sum())
            fn = int(((y_true == 1) & (y_pred == 0)).sum())

            recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
            fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

            row = {
                'disease': disease,
                'attribute': attr,
                'group': str(g),
                'n': int(len(sub)),
                'prevalence': float(sub['y_true'].mean()),
                'recall_tpr': float(recall) if pd.notna(recall) else np.nan,
                'fpr': float(fpr) if pd.notna(fpr) else np.nan,
                'precision': float(precision) if pd.notna(precision) else np.nan,
            }
            fair_rows.append(row)
            per_group.append(row)

        pg = pd.DataFrame(per_group)
        gap_rows.append({
            'disease': disease,
            'attribute': attr,
            'n_groups': int(pg.shape[0]),
            'tpr_gap': float(pg['recall_tpr'].max() - pg['recall_tpr'].min()),
            'fpr_gap': float(pg['fpr'].max() - pg['fpr'].min()),
            'precision_gap': float(pg['precision'].max() - pg['precision'].min()),
        })

fairness_detail_df = pd.DataFrame(fair_rows)
fairness_gap_df = pd.DataFrame(gap_rows).sort_values(['tpr_gap', 'fpr_gap'], ascending=False)

print('Fairness gap summary:')
if fairness_gap_df.empty:
    print('No groups met min_group_n threshold for fairness comparison.')
else:
    print(fairness_gap_df.to_string(index=False))

print('\nFairness detail (first 40 rows):')
if fairness_detail_df.empty:
    print('No detail rows.')
else:
    print(fairness_detail_df.head(40).to_string(index=False))

# Leakage audit (feature-level rules)
leak_rows = []

for disease, art in artifacts.items():
    feats = art.get('features', [])

    flags = []
    if any('label_next' in f for f in feats):
        flags.append('contains_label_next')
    if any(f.startswith('r15') for f in feats):
        flags.append('contains_future_wave_r15')
    if any(f.startswith('y_') or f == 'y' for f in feats):
        flags.append('contains_target')
    if f'hist_{disease}_prev' in feats:
        flags.append('contains_same_disease_history')

    leak_rows.append({
        'disease': disease,
        'n_features': int(len(feats)),
        'status': 'FAIL' if flags else 'PASS',
        'flags': ', '.join(flags) if flags else '',
    })

leakage_audit_df = pd.DataFrame(leak_rows)
print('\nLeakage audit:')
print(leakage_audit_df.to_string(index=False))



In [ ]:
# Explainability artifacts + frontend-ready sample payload
import warnings
import numpy as np
import pandas as pd

if 'artifacts' not in globals() or not artifacts:
    raise ValueError('Run training cell first: artifacts is missing.')

warnings.filterwarnings(
    'ignore',
    message='LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray',
    category=UserWarning,
)

# Optional SHAP
try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

exp_rows = []
local_rows = []

for disease, art in artifacts.items():
    lgb_models = art.get('lgb_models', [])
    if not lgb_models:
        continue

    feats = art.get('features', [])
    fi = np.mean([m.feature_importances_ for m in lgb_models], axis=0)
    fi_df = pd.DataFrame({'disease': disease, 'feature': feats, 'importance': fi})
    fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)

    top_global = fi_df.head(10).copy()
    exp_rows.extend(top_global.to_dict('records'))

    payload = eval_payloads.get(disease, None)
    if payload is None:
        continue

    p = payload['p_test'].values
    if len(p) == 0:
        continue

    idx = int(np.argmax(p))
    x_case = payload['X_test_model'].iloc[[idx]].copy()

    if HAS_SHAP:
        try:
            explainer = shap.TreeExplainer(lgb_models[0])
            sv = explainer.shap_values(x_case)
            if isinstance(sv, list):
                sv = sv[1]
            contrib = pd.DataFrame({
                'feature': x_case.columns,
                'shap_value': sv[0],
                'abs_shap': np.abs(sv[0]),
                'value': x_case.iloc[0].values,
            }).sort_values('abs_shap', ascending=False).head(6)

            for _, r in contrib.iterrows():
                local_rows.append({
                    'disease': disease,
                    'feature': r['feature'],
                    'value': float(r['value']),
                    'impact': float(r['shap_value']),
                    'direction': 'up' if r['shap_value'] > 0 else 'down',
                })
        except Exception:
            for _, r in fi_df.head(6).iterrows():
                local_rows.append({
                    'disease': disease,
                    'feature': r['feature'],
                    'value': np.nan,
                    'impact': float(r['importance']),
                    'direction': 'up',
                })
    else:
        for _, r in fi_df.head(6).iterrows():
            local_rows.append({
                'disease': disease,
                'feature': r['feature'],
                'value': np.nan,
                'impact': float(r['importance']),
                'direction': 'up',
            })

global_explain_df = pd.DataFrame(exp_rows)
local_explain_df = pd.DataFrame(local_rows)

print('Global top features per disease (top 10 each):')
print(global_explain_df.to_string(index=False))

print('\nLocal explanation sample (top case per disease):')
print(local_explain_df.to_string(index=False))

frontend_preview = []
for _, row in metrics_df.iterrows():
    disease = row['disease']
    payload = eval_payloads.get(disease, None)
    if payload is None or len(payload['p_test']) == 0:
        continue

    top_idx = int(np.argmax(payload['p_test'].values))
    top_prob = float(payload['p_test'].values[top_idx])

    loc = local_explain_df[local_explain_df['disease'] == disease].head(3)
    factors = []
    for _, fr in loc.iterrows():
        factors.append({
            'factor_name': str(fr['feature']),
            'impact': float(abs(fr['impact'])),
            'direction': str(fr['direction']),
            'value': None if pd.isna(fr['value']) else float(fr['value']),
        })

    frontend_preview.append({
        'disease': disease,
        'risk_probability': top_prob,
        'risk_level_rule': f"high if p >= {row['threshold']:.4f}",
        'metrics': {
            'F2': float(row['F2']),
            'PR-AUC': float(row['PR-AUC']),
            'ROC-AUC': float(row['ROC-AUC']),
        },
        'top_factors': factors,
    })

print('\nFrontend preview objects (first 2):')
print(frontend_preview[:2])

